In [0]:
select * from gizmobox.bronze.v_customers;

Databricks data profile. Run in Databricks to view.

In [0]:
%python
df = spark.sql("select * from gizmobox.bronze.v_customers")

In [0]:
%python
df.describe().show()

In [0]:
%python
dbutils.data.summarize(df)

In [0]:
select customer_id,max(created_timestamp),max(customer_name),max(date_of_birth),max(email),max(member_since),max(telephone) from gizmobox.bronze.v_customers
where customer_id is not null
group by customer_id ;

-- this is not ideal as we are not handling the case where there is no customer name, date of birth, email, member since, telephone and also getting data from diffrent rows lets use CTE to solve this problem

In [0]:
select customer_id, max(t.created_timestamp ) from gizmobox.bronze.v_customers t 
group  by customer_id
-- we want to keep the newest record for each customer

In [0]:
-- CTE 
with cte_max as (select customer_id, max(t.created_timestamp ) tmax from gizmobox.bronze.v_customers t 
group  by customer_id) select distinct(*) from cte_max inner join gizmobox.bronze.v_customers t on cte_max.customer_id = t.customer_id and cte_max.tmax = t.created_timestamp order by 
t.customer_id

In [0]:
DROP VIEW v_customers_distinct

In [0]:
create or replace  view gizmobox.bronze.v_customers_distinct as (
select distinct(*) from gizmobox.bronze.v_customers where customer_id is not null order by customer_id )

In [0]:
select * from gizmobox.bronze.v_customers_distinct

In [0]:
-- CTE 
with cte_max as (select customer_id, max(t.created_timestamp ) tmax from gizmobox.bronze.v_customers t 
group  by customer_id) select * from cte_max inner join gizmobox.bronze.v_customers_distinct t on cte_max.customer_id = t.customer_id and cte_max.tmax = t.created_timestamp order by 
t.customer_id

Cast 

In [0]:
with cte_max as (select customer_id, max(t.created_timestamp ) tmax from gizmobox.bronze.v_customers t 
group  by customer_id) 
select CAST (tt.created_timestamp AS TIMESTAMP) as created_timestamp,
 tt.customer_id,
 tt.customer_name ,
CAST (tt.date_of_birth AS DATE) as date_of_birth,
 tt.email as email,
 tt.telephone as telephone,
CAST (tt.member_since AS DATE) as member_since
from gizmobox.bronze.v_customers_distinct tt INNER JOIN cte_max on cte_max.customer_id = tt.customer_id and cte_max.tmax = tt.created_timestamp order by tt.customer_id

In [0]:
create table gizmobox.silver.customers 
with cte_max as (select customer_id, max(t.created_timestamp ) tmax from gizmobox.bronze.v_customers t 
group  by customer_id) 
select CAST (tt.created_timestamp AS TIMESTAMP) as created_timestamp,
 tt.customer_id,
 tt.customer_name ,
CAST (tt.date_of_birth AS DATE) as date_of_birth,
 tt.email as email,
 tt.telephone as telephone,
CAST (tt.member_since AS DATE) as member_since
from gizmobox.bronze.v_customers_distinct tt INNER JOIN cte_max on cte_max.customer_id = tt.customer_id and cte_max.tmax = tt.created_timestamp order by tt.customer_id

In [0]:
select * from gizmobox.silver.customers

In [0]:
describe  extended gizmobox.silver.customers

Tansform Payment table data 

In [0]:
select p.payment_id,p.order_id,p.payment_timestamp,p.payment_status,p.payment_method from gizmobox.bronze.payments p

we are trying to take the payment timestamp and breake it to date and time , add mapping for the payment status and write transformed data into silver schema
1- Success
2- Pending
3- Cancelled 
4- Failed


Payment method 
1- paypal
2- Credit Card
3- bank transfer 


In [0]:
select distinct ( p.payment_method) from gizmobox.bronze.payments p 

In [0]:
-- date_format(current_timestamp() ,'yyyy-MM-dd HH:mm:ss) it accepts two inputs date and the format we want to convert it to
select p.payment_id,p.order_id,p.payment_timestamp,date_format(p.payment_timestamp, 'yyyy-MM-dd') as payment_date ,date_format(p.payment_timestamp, 'HH:mm:ss') as payment_time,p.payment_status,p.payment_method from gizmobox.bronze.payments p

In [0]:
--  case when p.payment_status = 1 then 'paid' else 'unpaid' end as payment_status 
-- 1- Success
-- 2- Pending
-- 3- Cancelled 
-- 4- Failed
select p.payment_id,p.order_id,p.payment_timestamp,date_format(p.payment_timestamp, 'yyyy-MM-dd') as payment_date ,date_format(p.payment_timestamp, 'HH:mm:ss') as payment_time, payment_status as payment_status_code,
case p.payment_status
when 1 then 'Success'
when 2 then 'Pending'
when 3 then 'Cancelled'
else 'Failed'
end as payment_status_desc
,payment_method
, case p.payment_method
when 'Credit Card' then '1'
when 'PayPal' then '2'
when 'Bank Transfer' then '3'
 end as payment_method_code from gizmobox.bronze.payments p

In [0]:
create table gizmobox.silver.payments 
as 
--  case when p.payment_status = 1 then 'paid' else 'unpaid' end as payment_status 
-- 1- Success
-- 2- Pending
-- 3- Cancelled 
-- 4- Failed
select p.payment_id,p.order_id,p.payment_timestamp,date_format(p.payment_timestamp, 'yyyy-MM-dd') as payment_date ,date_format(p.payment_timestamp, 'HH:mm:ss') as payment_time, payment_status as payment_status_code,
case p.payment_status
when 1 then 'Success'
when 2 then 'Pending'
when 3 then 'Cancelled'
else 'Failed'
end as payment_status_desc
,payment_method
, case p.payment_method
when 'Credit Card' then '1'
when 'PayPal' then '2'
when 'Bank Transfer' then '3'
 end as payment_method_code from gizmobox.bronze.payments p

In [0]:
select * from gizmobox.silver.payments;


In [0]:
describe  extended gizmobox.silver.payments

In [0]:
-- regex
select regexp_replace('1934567890', '[0-5]', '-') as masked_phone_number


In [0]:
select * from gizmobox.bronze.v_memberships;


In [0]:
create table gizmobox.silver.membership as
select regexp_extract(path, '.*/([1-9]+)\\.png$', 1)  as custoner_id , content as membership_card from gizmobox.bronze.v_memberships

Pivoting : using rows to create colums based on groups 

In [0]:
select * from gizmobox.bronze.v_addresses

In [0]:
create table gizmobox.silver.addresses as select * from (select p.customer_id,p.address_line_1,
p.address_type,
p.city,
p.postcode,
p.state from gizmobox.bronze.v_addresses p )
 pivot (max(address_line_1) as address_line_1
 , max(city) as city
 , max(postcode) as postcode
 , max(state) as state for address_type in ('shipping','billing'))

In [0]:
select * from gizmobox.silver.addresses

In [0]:
select * from (select p.customer_id,p.address_line_1,
p.address_type,
p.city,
p.postcode,
p.state from gizmobox.bronze.v_addresses p )
 pivot (max(address_line_1) as address_line_1
 , max(city) as city
 , max(postcode) as postcode
 , max(state) as state for address_type in ( select distinct(address_type)  from gizmobox.bronze.v_addresses)



In [0]:

SELECT * FROM read_files('/Volumes/gizmobox/landing/operational_data/orders', format=>'json');


In [0]:
select * from gizmobox.bronze.v_orders;


JSON info extraction 

In [0]:
select value: items[0].item_id::int,*
from gizmobox.bronze.v_orders
-- this is not ideal so nect part

In [0]:
select  regexp_replace(value,'"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "\$1")') as fixed_value from gizmobox.bronze.v_orders

In [0]:
-- regex_replace
create or replace temp view tv_orders_fixed as select  regexp_replace(value,'"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "\$1"') as fixed_value from gizmobox.bronze.v_orders

In [0]:
select  * from tv_orders_fixed; 
-- schema of json
select schema_of_json(fixed_value), fixed_value from tv_orders_fixed limit 1


In [0]:
drop table order_json

In [0]:
create table gizmobox.silver.order_json as select from_json (fixed_value,'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value from tv_orders_fixed 

In [0]:
select * from gizmobox.silver.order_json

In [0]:
-- accessing elements from json
create or replace temp view tv_explode_order as select json_value.customer_id,  json_value.order_date, json_value.order_id, json_value.order_status, json_value.payment_method, json_value.total_amount, 
explode(array_distinct(json_value.items) ) from gizmobox.silver.order_json

In [0]:
drop table gizmobox.silver.orders

In [0]:
create table gizmobox.silver.orders as select t.order_id, t.order_date, t.order_status, t.payment_method, t.total_amount
,t.col.item_id as item_id,
t.col.name as item_name,
t.col.price as item_price,
t.col.quantity as item_quantity
,t.col.details.brand as item_brand,
t.col.details.color as item_color,
t.col.category as item_category,
t.customer_id
 from tv_explode_order t

Joins


In [0]:
create table gizmobox.gold.customers_address  as select c.customer_id,c.customer_name,c.date_of_birth,c.email,c.telephone,c.member_since,a.shipping_address_line_1,a.shipping_city,a.shipping_postcode,a.shipping_state,a.billing_address_line_1,a.billing_city,a.billing_postcode,a.billing_state
 from gizmobox.silver.addresses a  inner join 
gizmobox.silver.customers c on c.customer_id = a.customer_id


In [0]:
select * from gizmobox.gold.customers_address

aggregation functions


In [0]:
select * from gizmobox.silver.orders y 
where y.customer_id=5011

In [0]:
create table gizmobox.gold.orders_summary_monthly as 
select date_format(t.order_date,'yyyy-MM') as order_month, t.customer_id as customer, count(distinct(t.order_id)) as total_orders ,sum(t.item_price*t.item_quantity) as total_spent,sum(t.item_quantity) as total_qty from gizmobox.silver.orders t
group by order_month,customer
order by order_month

In [0]:
select * from gizmobox.gold.orders_summary_monthly

UDF - User Defined Function

In [0]:
-- return full name 
create or replace function gizmobox.default.get_fullname(firstname string, lastname string)
returns string
return initcap (firstname )|| ' ' || initcap(lastname);
select gizmobox.default.get_fullname('John','nSmith')

In [0]:
DESC FUNCTION EXTENDED gizmobox.default.get_fullname;

In [0]:
CREATE OR REPLACE FUNCTION gizmobox.default.get_payment_status(payment_status INT)
RETURNS STRING
RETURN CASE payment_status
         WHEN 1 THEN 'Success'
         WHEN 2 THEN 'Pending'
         WHEN 3 THEN 'Cancelled'
         WHEN 4 THEN 'Failed'
       END;

In [0]:
SELECT payment_id,
       order_id,
       CAST(date_format(payment_timestamp,'yyyy-MM-dd') AS DATE) AS payment_date,
       date_format(payment_timestamp,'HH:mm:ss') AS payment_time,
       gizmobox.default.get_payment_status(payment_status) AS payment_status,  
       payment_method
  FROM gizmobox.bronze.payments;

In [0]:
desc  function extended gizmobox.default.get_payment_status;

Higher Order Function ( Array ) 

In [0]:
CREATE OR REPLACE TEMPORARY VIEW order_items AS
SELECT * FROM 
VALUES
  (1, array('smartphone', 'laptop', 'monitor')),
  (2, array('tablet', 'headphones', 'smartwatch')),
  (3, array('keyboard', 'mouse'))
AS orders(order_id, items);
--  to be able to work with arrays

In [0]:
SELECT * FROM order_items;

##### 1. Convert all the item names to be UPPERCASE (TRANSFORM Function)

In [0]:
SELECT order_id,
       TRANSFORM(items, x -> UPPER(x)) AS upper_items
  FROM order_items;
  -- lambda function x -> function (x) 
  -- other higer order functions for arrays are transform ,FILTER, EXISTS, AGGREGATE

In [0]:
SELECT order_id,
       FILTER(items, x -> x.name LIKE '%smart%') AS smart_items
  FROM order_items;
  --  sample for Filter : it is filtering if the word smart is in there in SQL % is any charachter and _ is one character in regex * is any charachter and . is one 

In [0]:
CREATE OR REPLACE TEMP VIEW order_items AS
SELECT * FROM VALUES
  (1, array(
        named_struct('name', 'smartphone', 'price', 699),
        named_struct('name', 'laptop', 'price', 1199),
        named_struct('name', 'monitor', 'price', 399)
    )),
  (2, array(
        named_struct('name', 'tablet', 'price', 599),
        named_struct('name', 'headphones', 'price', 199),
        named_struct('name', 'smartwatch', 'price', 299)
    )),
  (3, array(
        named_struct('name', 'keyboard', 'price', 89),
        named_struct('name', 'mouse', 'price', 59)
    ))
AS orders(order_id, items);


In [0]:
SELECT order_id,
       TRANSFORM(items, x -> named_struct(
                                          'name', UPPER(x.name),
                                          'price', ROUND(x.price * 1.10, 2)
                                          )) items_with_tax
  FROM order_items
  --  Convert all the item names to be UPPERCASE & Add 10% TAX to each item (TRANSFORM Function)